In [17]:
# ======================================
# TRACEFI - Hybrid Deep Learning Model
# ======================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [18]:
# ======================================
# Step 1: Collect Parquet Files
# ======================================

DATASET_DIR = "./dataset"
ALLOWED_PREFIXES = ["DNS", "LDAP", "NTP", "UDP", "Syn"]

parquet_files = []

for file in os.listdir(DATASET_DIR):
    if file.endswith(".parquet"):
        if "UDPLag" in file:
            continue
        for prefix in ALLOWED_PREFIXES:
            if file.startswith(prefix):
                parquet_files.append(os.path.join(DATASET_DIR, file))
                break

print("Total parquet files:", len(parquet_files))

Total parquet files: 10


In [19]:
# ======================================
# Step 2: Load and Merge Dataset
# ======================================

dfs = [pd.read_parquet(file) for file in parquet_files]
df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", df.shape)


Combined dataset shape: (393775, 78)


In [20]:
# ======================================
# Step 3: Detect Label Column
# ======================================

possible_label_cols = ["label", "Label", "class", "Class"]

label_col = None
for col in possible_label_cols:
    if col in df.columns:
        label_col = col
        break

if label_col is None:
    raise ValueError("No label column detected!")

print("Detected label column:", label_col)

Detected label column: Label


In [21]:
# ======================================
# Step 4: Convert to Binary Labels
# ======================================

df["binary_label"] = df[label_col].apply(
    lambda x: 0 if str(x).lower() == "benign" else 1
)

print("Binary label distribution:")
print(df["binary_label"].value_counts())


Binary label distribution:
binary_label
1    322836
0     70939
Name: count, dtype: int64


In [22]:
# ======================================
# Step 5: Drop Leakage Columns
# ======================================

leakage_columns = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp"
]

df.drop(
    columns=[col for col in leakage_columns if col in df.columns],
    inplace=True
)



In [23]:
# ======================================
# Step 6: Handle NaN / Infinite
# ======================================

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Dataset shape after cleaning:", df.shape)

Dataset shape after cleaning: (393775, 79)


In [24]:
# ======================================
# Step 7: Separate Features and Target
# ======================================

X = df.drop(columns=["binary_label", label_col])
y = df["binary_label"]


In [25]:
# ======================================
# Step 8: Remove Non-Numeric Columns
# ======================================

non_numeric_cols = X.select_dtypes(include=["object", "category"]).columns
X = X.drop(columns=non_numeric_cols)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (393775, 77)


In [26]:
# ======================================
# Step 9: Feature Selection (Top 15)
# ======================================

selector = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

selector.fit(X, y)

importances = pd.Series(
    selector.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

TOP_N = 15
selected_features = importances.head(TOP_N).index.tolist()

X = X[selected_features]

print("Selected features:")
for f in selected_features:
    print("-", f)


Selected features:
- URG Flag Count
- Fwd Packet Length Mean
- Avg Fwd Segment Size
- Down/Up Ratio
- Packet Length Min
- Protocol
- Bwd Packet Length Min
- CWE Flag Count
- ACK Flag Count
- Fwd Packet Length Min
- Packet Length Mean
- Avg Bwd Segment Size
- Bwd Packet Length Mean
- Fwd Packet Length Max
- Avg Packet Size


In [27]:
# ======================================
# Step 10: Train-Test Split
# ======================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (315020, 15)
Test shape: (78755, 15)


In [28]:
# ======================================
# Step 11: Scaling (Required for DL)
# ======================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [29]:
# ======================================
# Step 12: Reshape for CNN
# ======================================

X_train_dl = X_train_scaled.reshape(
    (X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
)

X_test_dl = X_test_scaled.reshape(
    (X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
)


In [30]:
# ======================================
# Step 13: Compute Class Weights
# ======================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))

In [31]:
# ======================================
# Step 14: Build Hybrid Model
# ======================================

model = Sequential()

model.add(Conv1D(
    filters=32,
    kernel_size=3,
    activation='relu',
    input_shape=(X_train_dl.shape[1], 1)
))

model.add(MaxPooling1D(pool_size=2))
model.add(BatchNormalization())

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_1 (Conv1D)           (None, 13, 32)            128       
                                                                 
 max_pooling1d_1 (MaxPooling  (None, 6, 32)            0         
 1D)                                                             
                                                                 
 batch_normalization_1 (Batc  (None, 6, 32)            128       
 hNormalization)                                                 
                                                                 
 flatten (Flatten)           (None, 192)               0         
                                                                 
 dense_14 (Dense)            (None, 64)                12352     
                                                                 
 dropout_2 (Dropout)         (None, 64)               

In [32]:
# ======================================
# Step 15: Train Model
# ======================================

history = model.fit(
    X_train_dl,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    class_weight=class_weights_dict,
    verbose=1
)


Epoch 1/20
1969/1969 [==============================] - 31s 15ms/step - loss: 0.0614 - accuracy: 0.9855 - val_loss: 0.0421 - val_accuracy: 0.9902
Epoch 2/20
1969/1969 [==============================] - 25s 13ms/step - loss: 0.0413 - accuracy: 0.9896 - val_loss: 0.0343 - val_accuracy: 0.9912
Epoch 3/20
1969/1969 [==============================] - 33s 17ms/step - loss: 0.0387 - accuracy: 0.9898 - val_loss: 0.0348 - val_accuracy: 0.9912
Epoch 4/20
1969/1969 [==============================] - 31s 16ms/step - loss: 0.0370 - accuracy: 0.9900 - val_loss: 0.0402 - val_accuracy: 0.9903
Epoch 5/20
1969/1969 [==============================] - 29s 15ms/step - loss: 0.0434 - accuracy: 0.9898 - val_loss: 0.0331 - val_accuracy: 0.9901
Epoch 6/20
1969/1969 [==============================] - 29s 15ms/step - loss: 0.0374 - accuracy: 0.9884 - val_loss: 0.0325 - val_accuracy: 0.9904
Epoch 7/20
1969/1969 [==============================] - 29s 15ms/step - loss: 0.0373 - accuracy: 0.9899 - val_loss: 0.0308 -

In [33]:
# ======================================
# Step 16: Evaluate Model
# ======================================

y_pred_probs = model.predict(X_test_dl)
y_pred = (y_pred_probs > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_probs))

2462/2462 [==============================] - 16s 6ms/step

Confusion Matrix:
[[13887   301]
 [  376 64191]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98     14188
           1       1.00      0.99      0.99     64567

    accuracy                           0.99     78755
   macro avg       0.98      0.99      0.99     78755
weighted avg       0.99      0.99      0.99     78755


ROC-AUC Score: 0.9993650236207976


In [34]:
# ======================================
# Step 17: Save Model
# ======================================

model.save("tracefi_hybrid_model.keras")

print("\n✅ Hybrid model training completed successfully")


✅ Hybrid model training completed successfully


In [35]:
import pickle

# Save scaler
with open("tracefi_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save selected feature list
with open("tracefi_selected_features.pkl", "wb") as f:
    pickle.dump(selected_features, f)

# Save Keras model using .save (best practice)
model.save("tracefi_hybrid_model.h5")

print("✅ Model, scaler, and feature list saved successfully")

✅ Model, scaler, and feature list saved successfully


In [36]:
# ===============================
# Hybrid Model Evaluation
# ===============================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Get probabilities
y_prob_hybrid = model.predict(X_test_dl)

# Convert probabilities to class labels (binary threshold 0.5)
y_pred_hybrid = (y_prob_hybrid > 0.5).astype(int).ravel()

# Metrics
accuracy_h = accuracy_score(y_test, y_pred_hybrid)
precision_h = precision_score(y_test, y_pred_hybrid)
recall_h = recall_score(y_test, y_pred_hybrid)
f1_h = f1_score(y_test, y_pred_hybrid)
roc_auc_h = roc_auc_score(y_test, y_prob_hybrid)

print("Hybrid Model Performance:")
print(f"Accuracy  : {accuracy_h:.4f}")
print(f"Precision : {precision_h:.4f}")
print(f"Recall    : {recall_h:.4f}")
print(f"F1-score  : {f1_h:.4f}")
print(f"ROC-AUC   : {roc_auc_h:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hybrid))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hybrid))

2462/2462 [==============================] - 12s 5ms/step
Hybrid Model Performance:
Accuracy  : 0.9914
Precision : 0.9953
Recall    : 0.9942
F1-score  : 0.9948
ROC-AUC   : 0.9994

Confusion Matrix:
[[13887   301]
 [  376 64191]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98     14188
           1       1.00      0.99      0.99     64567

    accuracy                           0.99     78755
   macro avg       0.98      0.99      0.99     78755
weighted avg       0.99      0.99      0.99     78755

